In [1]:
%matplotlib inline

In [2]:

import warnings
import os
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.style as style
import matplotlib.cm as cm
from scipy.io import loadmat
import scipy
from sklearn.manifold import LocallyLinearEmbedding
import nilearn as nl             # pip: nilearn==0.10.0
from nilearn.maskers import NiftiMasker
import nibabel as nib
import nltools as nlt
from matplotlib.colors import ListedColormap




In [3]:

################################################################################
# Archetypal Analysis        ####################################################
################################################################################
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

def furthest_sum(X, n_archetypes, random_state=None):
    """
    Memory-safe FurthestSum initialization.
    
    Parameters
    ----------
    X : np.ndarray, shape (n_features, n_samples)
        Data matrix.
    n_archetypes : int
        Number of archetypes to select.
    random_state : int or None
        Seed for reproducibility.
    
    Returns
    -------
    selected : list of int
        Selected indices for initialization.
    """
    rng = np.random.default_rng(random_state)
    n_samples = X.shape[1]
    selected = [rng.integers(0, n_samples)]

    for _ in range(1, n_archetypes):
        dists = np.zeros(n_samples, dtype=np.float32)
        for i in range(n_samples):
            dists[i] = sum(np.linalg.norm(X[:, i] - X[:, j]) for j in selected)
        dists[selected] = -np.inf  # don't re-select
        selected.append(np.argmax(dists))

    return selected
    
class MultiSubjectArchetypalAnalysis(BaseEstimator, TransformerMixin):
    def __init__(self, n_archetypes=3, tmax=20, iterations=10, init_method='furthest', random_state=42, verbose=True):
        self.n_archetypes = n_archetypes
        self.tmax = tmax
        self.iterations = iterations
        self.init_method = init_method
        self.random_state = random_state
        self.verbose = verbose
        self.Z_ = None
        self.A_ = None

    def fit(self, Xs):
        """
        Fit MS-AA to a list of subject matrices.

        Parameters
        ----------
        Xs : list of np.ndarray
            Each with shape (n_features, n_timepoints)
        """
        # Convert to float32 to reduce memory usage
        Xs = [X.astype(np.float32) for X in Xs]
        n_subjects = len(Xs)
        n_features, _ = Xs[0].shape
        k = self.n_archetypes

        # Concatenate all subjects once
        X_concat = np.concatenate(Xs, axis=1).astype(np.float32)

        # Initialize archetypes
        if self.init_method == 'furthest':
            init_indices = furthest_sum(X_concat, k, random_state=self.random_state)
            B = np.zeros((X_concat.shape[1], k), dtype=np.float32)
            for i, idx in enumerate(init_indices):
                B[idx, i] = 1.0
        elif self.init_method == 'random':
            rng = np.random.default_rng(self.random_state)
            init_indices = rng.choice(X_concat.shape[1], size=k, replace=False)
            B = np.zeros((X_concat.shape[1], k), dtype=np.float32)
            for i, idx in enumerate(init_indices):
                B[idx, i] = 1.0
        else:  # identity
            B = np.eye(X_concat.shape[1], k, dtype=np.float32)

        Z = X_concat @ B
        A_subjects = [np.zeros((k, X.shape[1]), dtype=np.float32) for X in Xs]

        for iteration in range(self.iterations):
            # Update per-subject A
            for i, X in enumerate(Xs):
                A_subjects[i] = self._computeA(X, Z, self.tmax)

            # Concatenate for B update
            A_concat = np.concatenate(A_subjects, axis=1).astype(np.float32)
            B = self._computeB(X_concat, A_concat, self.tmax)
            Z = X_concat @ B

            if self.verbose:
                rss_total = sum(self._rss(Xs[i], A_subjects[i], Z) for i in range(n_subjects))
                print(f"Iteration {iteration + 1}, Total RSS: {rss_total:.4f}")

        self.Z_ = Z
        self.A_ = A_subjects
        return self

    def transform(self, X):
        return self._computeA(X.astype(np.float32), self.Z_, self.tmax)

    def inverse_transform(self, A):
        return self.Z_ @ A

    def archetypes(self):
        return self.Z_

    def subject_weights(self):
        return self.A_

    def _computeA(self, X, Z, tmax):
        m, n = X.shape
        k = self.n_archetypes
        A = np.zeros((k, n), dtype=np.float32)
        A[0, :] = 1.0

        for t in range(tmax):
            G = 2.0 * ((Z.T @ Z) @ A - Z.T @ X)
            argmins = np.argmin(G, axis=0)
            e = np.zeros_like(G)
            e[argmins, np.arange(n)] = 1.0
            A += 2.0 / (t + 2.0) * (e - A)
        return A

    def _computeB(self, X, A, tmax):
        k, n = A.shape
        B = np.zeros((n, k), dtype=np.float32)
        B[0, :] = 1.0

        for t in range(tmax):
            t1 = X.T @ (X @ B) @ (A @ A.T)
            t2 = X.T @ (X @ A.T)
            G = 2.0 * (t1 - t2)
            argmins = np.argmin(G, axis=0)
            e = np.zeros_like(G)
            e[argmins, np.arange(k)] = 1.0
            B += 2.0 / (t + 2.0) * (e - B)
        return B

    def _rss(self, X, A, Z):
        return np.linalg.norm(X - Z @ A)

In [5]:

def nii2cmu(nifti_file, mask_file=None):
    '''
    inputs:
      nifti_file: a filename of a .nii or .nii.gz file to be converted into
                  CMU format

      mask_file: a filename of a .nii or .nii.gz file to be used as a mask; all
                 zero-valued voxels in the mask will be ignored in the CMU-
                 formatted output.  If ignored or set to None, no voxels will
                 be masked out.

    outputs:
      Y: a number-of-timepoints by number-of-voxels numpy array containing the
         image data.  Each row of Y is an fMRI volume in the original nifti
         file.

      R: a number-of-voxels by 3 numpy array containing the voxel locations.
         Row indices of R match the column indices in Y.
    '''
    def fullfact(dims):
        '''
        Replicates MATLAB's fullfact function (behaves the same way)
        '''
        vals = np.asmatrix(range(1, dims[0] + 1)).T
        if len(dims) == 1:
            return vals
        else:
            aftervals = np.asmatrix(fullfact(dims[1:]))
            inds = np.asmatrix(np.zeros((np.prod(dims), len(dims))))
            row = 0
            for i in range(aftervals.shape[0]):
                inds[row:(row + len(vals)), 0] = vals
                inds[row:(row + len(vals)), 1:] = np.tile(aftervals[i, :], (len(vals), 1))
                row += len(vals)
            return inds

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        if type(nifti_file) == str:
            img = nib.load(nifti_file)
        elif type(nifti_file) == nib.nifti1.Nifti1Image:
            img = nifti_file
        else:
            raise ValueError('nifti_file must be a filename or nibabel Nifti1Image object')

        mask = NiftiMasker(mask_strategy='background')
        if mask_file is None:
            mask.fit(nifti_file)
        else:
            mask.fit(mask_file)

    hdr = img.header
    S = img.get_sform()
    vox_size = hdr.get_zooms()
    im_size = img.shape

    if len(img.shape) > 3:
        N = img.shape[3]
    else:
        N = 1

    Y = np.float32(mask.transform(nifti_file)).copy()
    vmask = np.nonzero(np.array(np.reshape(mask.mask_img_.dataobj, (1, np.prod(mask.mask_img_.shape)), order='C')))[1]
    vox_coords = fullfact(img.shape[0:3])[vmask, ::-1]-1

    R = np.array(np.dot(vox_coords, S[0:3, 0:3])) + S[:3, 3]

    return {'Y': Y, 'R': R}

def cmu2nii(Y, R, template=None):
    '''
    inputs:
      Y: a number-of-timepoints by number-of-voxels numpy array containing the
         image data.  Each row of Y is an fMRI volume in the original nifti
         file.

      R: a number-of-voxels by 3 numpy array containing the voxel locations.
         Row indices of R match the column indices in Y.

      template: a filename of a .nii or .nii.gz file to be used as an image
                template.  Header information of the outputted nifti images will
                be read from the header file.  If this argument is ignored or
                set to None, header information will be inferred based on the
                R array.

    outputs:
      nifti_file: a filename of a .nii or .nii.gz file to be converted into
                  CMU format

      mask_file: a filename for a .nii or .nii.gz file to be used as a mask; all
                 zero-valued voxels in the mask will be ignored in the CMU-
                 formatted output

    outputs:
      img: a nibabel Nifti1Image object containing the fMRI data
    '''
    Y = np.array(Y, ndmin=2)
    img = nib.load(template)
    S = img.affine
    locs = np.array(np.dot(R - S[:3, 3], np.linalg.inv(S[0:3, 0:3])), dtype='int')

    data = np.zeros(tuple(list(img.shape)[0:3]+[Y.shape[0]]))

    # loop over data and locations to fill in activations
    for i in range(Y.shape[0]):
        for j in range(R.shape[0]):
            data[locs[j, 0], locs[j, 1], locs[j, 2], i] = Y[i, j]

    return nib.Nifti1Image(data, affine=img.affine)
    
    
# def cmu2nii(activations, locations, template):
#     image = nib.load(template)
#     sform = image.get_sform()
#     coords = np.round(np.array(
#         np.dot(locations - sform[:3, 3],
#                np.linalg.inv(sform[0:3, 0:3]))
#     )).astype(int)
#     data = np.zeros(image.shape[0:3] + (activations.shape[0],))

#     for i in range(activations.shape[0]):
#         for j in range(locations.shape[0]):
#             x, y, z = coords[j, 0], coords[j, 1], coords[j, 2]
#             data[x, y, z, i] = activations[i, j]

#     return nib.Nifti1Image(data, affine=image.get_sform())

def node_labels(centers, widths, networks_cmu):
    labels = []
    for c, w in zip(centers, widths):
        r = rbf(networks_cmu['R'], c, w)

        label_weights = [sum(r[networks_cmu['Y'].ravel() == i]) for i in range(1, len(network_codes) + 1)]
        labels.append(np.argmax(label_weights) + 1)

    return pd.DataFrame({'code': labels, 'Network': [list(lookup_table.values())[i - 1] for i in labels]})

def rbf(R, center, width):
    return np.exp(-np.sum((R - center) ** 2, axis=1) / width)

def purity(y_clust,y_class):

    size_clust = np.max(y_clust)
    len_clust = len(y_clust)
    clusters_labels_try = [None] * size_clust

    for i in range(len_clust):
        index = y_clust[i]
    
        if clusters_labels_try[index-1] is None:
            clusters_labels_try[index-1] = y_class[i]

        else:
            clusters_labels_try[index-1] = np.hstack((clusters_labels_try[index-1], y_class[i]))
    purity = 0
    for c in clusters_labels_try:
        c = c.astype(int)
        if isinstance(c, np.int64):
    
            c = np.array([c]) 
        y = np.bincount(c) #I find occurrences of the present elements
        maximum = np.max(y) #I take the item more frequently
        purity += maximum

    purity = purity/len_clust

    return purity

# Pie Man Data

## Load the data here

The data we're using for this, to start, is fMRI data from particpants scanned while being told a story in different listening conditions.  The story was told either intact, or scrambled by the paragraph, or scrambled by the word, or just while the participant was quietly resting.  

The fMRI data was then reduced using Hierarchicaly topographic factor analysis (HTFA which is a Bayesian factor analysis model designed to analyze brain network dynamics in multi-subject datasets. It builds upon Topographic Factor Analysis (TFA), extending it to incorporate data from multiple subjects, allowing for the identification of shared brain network hubs and the comparison of individual subject's brain activity against a global template. In this case, the >100,000 voxels was reduced to 700 nodes with varying widths.




In [7]:
pieman_name = '../data/raw/pieman_data.mat'
pieman_data = loadmat(pieman_name)
pieman_conds = ['intact', 'paragraph', 'word', 'rest']

In [8]:
posterior = loadmat('../data/raw/pieman_posterior_K700.mat')
centers = posterior['posterior']['centers'][0][0][0][0][0]
widths = np.array(list(posterior['posterior']['widths'][0][0][0][0][0][:, 0].T))

These lines of code separate the participant data by listening conditions (and skips over one participant with an incorrect number of timepoints in the paragraph condition)

In [9]:
data = []
conds = []
for c in pieman_conds:
    if c == 'paragraph':
        next_data = list(map(lambda i: pieman_data[c][:, i][0], np.where(np.arange(pieman_data[c].shape[1]) != 3)[0]))
    else:
        next_data = list(map(lambda i: pieman_data[c][:, i][0], np.arange(pieman_data[c].shape[1])))
    data.extend(next_data)
    conds.extend([c]*len(next_data))


In [10]:
conds_array = np.array(conds)

First, lets just play with some of the intact listening condition data:

In [11]:
intact_list = [data[i] for i in np.where(conds_array=='intact')[0]]
intact_array = np.array(intact_list)

In [12]:
paragraph_list = [data[i] for i in np.where(conds_array=='paragraph')[0]]
paragraph_array = np.array(paragraph_list)

In [13]:
word_list = [data[i] for i in np.where(conds_array=='word')[0]]
word_array = np.array(word_list)

In [14]:
rest_list = [data[i] for i in np.where(conds_array=='rest')[0]]
rest_array = np.array(rest_list)

In [15]:
model = MultiSubjectArchetypalAnalysis(n_archetypes=5, init_method='furthest', iterations=10)
model.fit(intact_list)
Z = model.archetypes()
A_list = model.subject_weights()

Iteration 1, Total RSS: 18412.8073
Iteration 2, Total RSS: 17909.9641
Iteration 3, Total RSS: 17810.2178
Iteration 4, Total RSS: 17806.5499
Iteration 5, Total RSS: 17798.4993
Iteration 6, Total RSS: 17726.3505
Iteration 7, Total RSS: 17840.8738
Iteration 8, Total RSS: 17747.1191
Iteration 9, Total RSS: 17739.0131
Iteration 10, Total RSS: 17670.3302


In [16]:
archs = MultiSubjectArchetypalAnalysis(n_archetypes=5,iterations=10,tmax=300) #only doing 10 iterations now to save on time

archs.fit([intact_list[0]])

Z=(archs.archetypes())

for i in np.arange(5):
  plt.plot(Z[:, i]+ 2*i)


Iteration 1, Total RSS: 467.0906
Iteration 2, Total RSS: 453.5499
Iteration 3, Total RSS: 450.4449
Iteration 4, Total RSS: 448.7371
Iteration 5, Total RSS: 447.5191
Iteration 6, Total RSS: 446.6483
Iteration 7, Total RSS: 446.0898
Iteration 8, Total RSS: 445.7946
Iteration 9, Total RSS: 445.6337
Iteration 10, Total RSS: 445.5383
